In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## MODEL-CALL LIMIT MIDDLEWARE

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
)

In [7]:
config={"configurable":{"thread_id":"test-1"}}

In [10]:
from langchain.messages import HumanMessage
response=agent.invoke({"messages":[HumanMessage(content="Hello, my name is Arun")]},config=config)

In [11]:
response

{'messages': [HumanMessage(content='Hello, my name is Arun', additional_kwargs={}, response_metadata={}, id='dea3bba4-a164-402e-9723-660aa2e1a11f'),
  AIMessage(content="Hello Arun, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 42, 'total_tokens': 69, 'completion_time': 0.026433907, 'completion_tokens_details': None, 'prompt_time': 0.003752478, 'prompt_tokens_details': None, 'queue_time': 0.355275278, 'total_time': 0.030186385}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f36fb-eaf7-72b2-b745-a79101b0a27a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 27, 'total_tokens': 69})]}

In [13]:
from langchain.messages import HumanMessage
response=agent.invoke({"messages":[HumanMessage(content="Hello, multiply 2 by 3")]},config=config)

In [14]:
response

{'messages': [HumanMessage(content='Hello, my name is Arun', additional_kwargs={}, response_metadata={}, id='dea3bba4-a164-402e-9723-660aa2e1a11f'),
  AIMessage(content="Hello Arun, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 42, 'total_tokens': 69, 'completion_time': 0.026433907, 'completion_tokens_details': None, 'prompt_time': 0.003752478, 'prompt_tokens_details': None, 'queue_time': 0.355275278, 'total_time': 0.030186385}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f36fb-eaf7-72b2-b745-a79101b0a27a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 27, 'total_tokens': 69}),
  HumanMessage(content='Hello, multiply 2 by 3', additional_kwargs={}, respon

In [15]:
from langchain.messages import HumanMessage
response=agent.invoke({"messages":[HumanMessage(content="Hello, multiply divide 10 by 2")]},config=config)

In [16]:
response

{'messages': [HumanMessage(content='Hello, my name is Arun', additional_kwargs={}, response_metadata={}, id='dea3bba4-a164-402e-9723-660aa2e1a11f'),
  AIMessage(content="Hello Arun, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 42, 'total_tokens': 69, 'completion_time': 0.026433907, 'completion_tokens_details': None, 'prompt_time': 0.003752478, 'prompt_tokens_details': None, 'queue_time': 0.355275278, 'total_time': 0.030186385}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6a1eabf260', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f36fb-eaf7-72b2-b745-a79101b0a27a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 27, 'total_tokens': 69}),
  HumanMessage(content='Hello, multiply 2 by 3', additional_kwargs={}, respon

In [37]:
from langchain.tools import tool
@tool
def check_weather(query:str)->str:
    """Search the live weather of any location you want"""
    return f"The weather at {query} is sunny"

@tool
def check_time()->str:
    """Returns the current time of the day"""
    return f"The current time is: 4:30 PM"

@tool
def check_hotels(query:str)->str:
    """Check the available hotels in any city"""
    return f"The hotels in {query} are Grand Hyaatt, Regency, Mediterranenan, Cinepolis, Myesco"

In [41]:
# llm=ChatGroq(model="llama-3.1-8b-instant")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[check_weather,check_time,check_hotels],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=5,
            run_limit=1,
            exit_behavior="end",
        ),
    ],
)

In [42]:
config={"configurable":{"thread_id":"test-2"}}

In [43]:
from langchain.messages import HumanMessage
try:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Check weather in Italy, check current time, then check hotels in Italy."
                )
            ]
        },
        config=config,
    )

    print(response)
    print(response["messages"][-1].content)

except Exception as e:
    print(type(e).__name__)
    print(e)

{'messages': [HumanMessage(content='Check weather in Italy, check current time, then check hotels in Italy.', additional_kwargs={}, response_metadata={}, id='d6b4d2c1-f426-40bd-a9b0-ce8a5a459934'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 449, 'prompt_tokens': 182, 'total_tokens': 631, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dybko3rQCdbq4blyosDyzM2i8v972', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f372d-d284-7883-9a54-02fb3f7d4400-0', tool_calls=[{'name': 'check_weather', 'args': {'query': 'Italy'}, 'id': 'call_baJTyEKIsjfYH9F3FHjvfs4n', 'type': 'tool_call'}, {'name': 'check_time', 'args': 